# Muon variants: loss vs wall-clock and bpb vs LR (all depths)

Parses the nanochat training logs from the sweep (`logs/slurm/sweep/*.out`). Depths with no finished runs yet are skipped, so this can be re-run as the sweep progresses.

- **Wall-clock** = nanochat's own `total time` counter: pure training step time, excluding the first 10 steps, evals and compile/startup.
- **Wall-clock plots**: per depth, each variant at its best LR (lowest final val bpb among finished runs). Set `LR_MODE` to a number to pin one LR instead.
- **bpb vs LR plot**: final val bpb against `--matrix-lr`, one line per variant, one panel per depth.

In [ ]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DEPTHS = [8, 12, 16]
LR_MODE = "best"        # "best" = best final val bpb per (depth, variant); or a number like 0.02 to pin one LR for all
SHOW_ALL_LRS = False    # in the wall-clock plots, also draw the non-selected LRs as faint lines
LOSS_ZOOM, BPB_ZOOM = 1.0, 0.25   # y-range above the best final value in the wall-clock plots

LABELS = {
    "eq":    "MuonEq (nanochat default)",
    "eq_m2": "MuonEq + Muon$^2$",
    "m2":    "Muon$^2$",
    "base":  "Muon (no EQ, no Muon$^2$)",
}
COLORS = {"eq": "tab:blue", "eq_m2": "tab:red", "m2": "tab:green", "base": "tab:gray"}

In [ ]:
FILE_RE  = re.compile(r"sweep_d(\d+)_(eq_m2|eq|m2|base)_lr([0-9.eE+-]+?)-(\d+)\.out$")
STEP_RE  = re.compile(r"^step (\d+)/(\d+) .*?\| loss: ([0-9.]+) .*?\| dt: ([0-9.]+)ms .*?total time: ([0-9.]+)m", re.M)
VAL_RE   = re.compile(r"^Step (\d+) \| Validation bpb: ([0-9.]+)", re.M)

# Find every logs/slurm/sweep dir at or above the cwd (the scripts may have been launched from a subdir like jobs/)
log_dirs = [p / "logs" / "slurm" / "sweep" for p in [Path.cwd(), *Path.cwd().parents]]
log_dirs = [d for d in log_dirs if d.is_dir()]
print("log dirs:", [str(d) for d in log_dirs])

runs = {}   # (depth, variant, lr) -> dict; if a tag has several logs keep the one with the most steps
for d in log_dirs:
    for path in d.glob("*.out"):
        m = FILE_RE.search(path.name)
        if not m or int(m.group(1)) not in DEPTHS:
            continue
        text = path.read_text(errors="replace")
        steps = STEP_RE.findall(text)
        if not steps:
            continue
        train = pd.DataFrame(steps, columns=["step", "n_iters", "loss", "dt_ms", "time_min"]).astype(float)
        val = pd.DataFrame(VAL_RE.findall(text), columns=["step", "val_bpb"]).astype(float)
        # wall-clock (training time only) at each eval step; the last eval comes after the final train step
        val["time_min"] = np.interp(val["step"], train["step"], train["time_min"])
        n_iters = int(train["n_iters"].iloc[0])
        key = (int(m.group(1)), m.group(2), float(m.group(3)))
        if key not in runs or len(train) > len(runs[key]["train"]):
            runs[key] = dict(train=train, val=val, path=path, n_iters=n_iters,
                             finished=len(train) >= n_iters and len(val) > 0)

rows = []
for (depth, v, lr), r in sorted(runs.items()):
    rows.append(dict(depth=depth, variant=v, lr=lr, finished=r["finished"],
                     final_val_bpb=r["val"]["val_bpb"].iloc[-1] if len(r["val"]) else np.nan,
                     final_train_loss=r["train"]["loss"].iloc[-1],
                     train_min=r["train"]["time_min"].iloc[-1],
                     median_ms_per_step=r["train"]["dt_ms"].iloc[10:].median()))
summary = pd.DataFrame(rows)
avail = [int(d) for d in sorted(summary[summary.finished].depth.unique())] if len(summary) else []
print("depths with finished runs:", avail)
print("finished runs per depth:", summary[summary.finished].groupby("depth").size().to_dict() if len(summary) else {})
summary

In [ ]:
# Pick one LR per (depth, variant)
chosen = {}
for depth in avail:
    for v in LABELS:
        s = summary[(summary.depth == depth) & (summary.variant == v) & summary.finished]
        if s.empty:
            print(f"d{depth}: no finished runs for {v}"); continue
        if LR_MODE == "best":
            chosen[(depth, v)] = s.loc[s.final_val_bpb.idxmin(), "lr"]
        elif float(LR_MODE) in set(s.lr):
            chosen[(depth, v)] = float(LR_MODE)
        else:
            print(f"d{depth}: {v} has no finished run at lr={LR_MODE}")

sel = summary[summary.apply(lambda r: chosen.get((r.depth, r.variant)) == r.lr, axis=1)]
sel.set_index(["depth", "variant"]).sort_index()

In [ ]:
def plot_wallclock(depth):
    vs = [v for v in LABELS if (depth, v) in chosen]
    if not vs:
        return
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
    for v in vs:
        lr = chosen[(depth, v)]
        r = runs[(depth, v, lr)]
        label = f"{LABELS[v]} (lr={lr:g})"
        axes[0].plot(r["train"]["time_min"], r["train"]["loss"], color=COLORS[v], lw=1.5, label=label)
        axes[1].plot(r["val"]["time_min"], r["val"]["val_bpb"], color=COLORS[v], lw=1.5, marker="o", ms=3, label=label)
        if SHOW_ALL_LRS:
            for (d2, v2, lr2), r2 in runs.items():
                if d2 == depth and v2 == v and lr2 != lr and r2["finished"]:
                    axes[0].plot(r2["train"]["time_min"], r2["train"]["loss"], color=COLORS[v], lw=0.6, alpha=0.25)
                    axes[1].plot(r2["val"]["time_min"], r2["val"]["val_bpb"], color=COLORS[v], lw=0.6, alpha=0.25)

    # zoom the y-axes onto the interesting part (drop the initial high-loss transient)
    fl = min(runs[(depth, v, chosen[(depth, v)])]["train"]["loss"].iloc[-1] for v in vs)
    fb = min(runs[(depth, v, chosen[(depth, v)])]["val"]["val_bpb"].iloc[-1] for v in vs)
    axes[0].set_ylim(fl - 0.05, fl + LOSS_ZOOM)
    axes[1].set_ylim(fb - 0.05, fb + BPB_ZOOM)

    axes[0].set_title(f"Training loss (EMA-smoothed), depth {depth}")
    axes[0].set_ylabel("train loss (nats/token)")
    axes[1].set_title(f"Validation bpb, depth {depth}")
    axes[1].set_ylabel("val bpb")
    for ax in axes:
        ax.set_xlabel("training wall-clock (min)")
        ax.grid(alpha=0.3)
    axes[0].legend(frameon=False, fontsize=9)
    plt.tight_layout()
    plt.show()

for depth in avail:
    plot_wallclock(depth)

In [ ]:
# Final val bpb vs matrix LR, one panel per depth (star = best LR for that variant)
fin = summary[summary.finished]
if fin.empty:
    print("no finished runs yet")
else:
    fig, axes = plt.subplots(1, len(avail), figsize=(5.2 * len(avail), 4.4), squeeze=False)
    for ax, depth in zip(axes[0], avail):
        for v in LABELS:
            s = fin[(fin.depth == depth) & (fin.variant == v)].sort_values("lr")
            if s.empty:
                continue
            ax.plot(s.lr, s.final_val_bpb, color=COLORS[v], marker="o", ms=4, lw=1.5, label=LABELS[v])
            b = s.loc[s.final_val_bpb.idxmin()]
            ax.plot([b.lr], [b.final_val_bpb], color=COLORS[v], marker="*", ms=14, mec="k", mew=0.6)
        lrs = sorted(fin[fin.depth == depth].lr.unique())
        ax.set_xscale("log")
        ax.set_xticks(lrs)
        ax.set_xticklabels([f"{x:g}" for x in lrs])
        ax.minorticks_off()
        ax.set_title(f"depth {depth}")
        ax.set_xlabel("--matrix-lr")
        ax.grid(alpha=0.3)
    axes[0][0].set_ylabel("final val bpb")
    axes[0][0].legend(frameon=False, fontsize=9)
    plt.tight_layout()
    plt.show()

    # compact table of the same numbers: rows = (depth, variant), columns = LR
    display(fin.pivot_table(index=["depth", "variant"], columns="lr", values="final_val_bpb").round(4))